# **SPECTRA training notebook**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import argparse
import yaml
import pickle
import torch
import networkx as nx
import scanpy as sc
import numpy as np
import pandas as pd
import warnings
import os

# Suppress annoying warnings for a clean console
warnings.filterwarnings("ignore")

In [3]:
# torch device setting
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(torch.version.cuda)
    print(torch.cuda.get_device_name())
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

12.1
NVIDIA A100 80GB PCIe
Using device: cuda


In [35]:
from spectra.utils import set_seed
from spectra.data import data_preprocessing, build_model_dataloaders_perts_split, compute_weights, compute_weights_enhanced
from spectra.model import SPECTRA
from spectra.training import train

### **Load config file**

In [5]:
config_path = '../configs/rpe1_config.yaml'

with open(config_path, 'r') as file:
    config = yaml.safe_load(file)
print(yaml.dump(config, indent=2, sort_keys=False, default_flow_style=False))

project:
  name: SPECTRA_rpe1
  seed: 42
  deterministic: false
  use_wandb: true
data:
  adata_path: /group/sottoriva/michele.calabro/SPECTRA/data/rep_rpe1/replogle_rpe1_standard_filtered.h5ad
  gene_list_path: /group/sottoriva/michele.calabro/SPECTRA/data/rep_rpe1/gene_list.txt
  network_path: /group/sottoriva/michele.calabro/SPECTRA/data/rep_rpe1/rpe_network.tsv
  scgpt_embeddings_path: /group/sottoriva/michele.calabro/SPECTRA/data/rep_rpe1/scGPT_embeddings_all_genes.pkl
  gene_weights_path: /group/sottoriva/michele.calabro/SPECTRA/data/rep_rpe1/gene_weights-rpe1.pkl
  split_path: /group/sottoriva/michele.calabro/SPECTRA/data/rep_rpe1/ref_b_matched_perturbation_splits.json
model:
  architecture: wilconxon_vsrest_enhanced
  conv_type: FAGCN
  residual_weight: 0.8
  n_channels: 48
  dropout_p: 0.17
  num_node_features: 1
training:
  batch_size: 24
  lr: 0.001
  n_epochs: 20
  alpha: 2.4
  beta: 1.0e-05
  gamma: 2.4
  test_ratio: 0.2
  val_ratio: 0.1
  weights_folder_path: /group/sotto

In [6]:
# Initialization & Seeding - set it to deterministic for reproducibility
set_seed(seed=config['project']['seed'], deterministic=config['project']['deterministic'])

### **Load files**

In [20]:
# Load adata
adata = sc.read_h5ad(config['data']['adata_path'])
#adata.raw = adata.copy()
adata = data_preprocessing(adata,
    logtransform=True, 
    min_cells_per_pert=50)
adata

AnnData object with n_obs × n_vars = 209586 × 8749
    obs: 'batch', 'target_gene', 'gene_id', 'transcript', 'gene_transcript', 'guide_id', 'percent_mito', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'disease', 'cancer', 'cell_line', 'sex', 'age', 'perturbation', 'organism', 'perturbation_type', 'tissue_type', 'ncounts', 'ngenes', 'nperts', 'percent_ribo', 'celltype', 'n_genes'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells', 'n_cells', 'gene_name'
    uns: 'log1p'

In [21]:
# scGPT gene embeddings
with open(config['data']['scgpt_embeddings_path'], "rb") as f:
    scgpt_dict = pickle.load(f)

In [22]:
# load GRN
network_data = pd.read_csv(config['data']['network_path'], sep='\t')
G = nx.DiGraph()
for _, edge in network_data.iterrows():
    G.add_edge(edge['source'], edge['target'], weight=edge['weight'])

G.remove_nodes_from([n for n in G.nodes if n not in scgpt_dict])
grn_genes = set(G.nodes)

Number of nodes: 4889
Number of edges: 210278


In [25]:
# Filter adata to match final network genes (gene_list)
gene_list = grn_genes & set(adata.var_names)

if grn_genes != gene_list:
    print('WARNING: some genes in the provided gene list are not included in the grn, or the gene embeddings are missing; filtering them out...')
adata = adata[:, adata.var_names.isin(gene_list)]
G = G.subgraph(gene_list).copy()

num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
print("Number of nodes:", num_nodes)
print("Number of edges:", num_edges)

Number of nodes: 4835
Number of edges: 205399


In [26]:
# Filter out perturbations that aren't in the gene list
perturbations = list(adata.obs['target_gene'].unique())
perturbations.remove('non-targeting')
perts_not_included = list({
    pert for pert in perturbations
    if any(single_pert not in gene_list for single_pert in pert.split('+'))
})
if len(perts_not_included)>0:
    print('WARNING: some perturbed genes are not included in the GRN. Filtering these perturbation samples out...')
adata = adata[~adata.obs['target_gene'].isin(perts_not_included)].copy()

None


### **Edge indices, embeddings matrix**
Other preparation steps fro SPECTRA

In [27]:
# sanity check
assert set(G.nodes) == set(adata.var_names), "Nodes in G and adata.var_names differ!"

In [28]:
# Map Edge Index
gene_to_idx = {node: i for i, node in enumerate(adata.var_names)}
edges = [(gene_to_idx[u], gene_to_idx[v]) for u, v in G.edges()]
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

In [29]:
# Build embedding matrix
scgpt_dict = {gene_to_idx[k]: v for k, v in scgpt_dict.items() if k in gene_to_idx}
scgpt_dim = len(next(iter(scgpt_dict.values())))
embedding_matrix = torch.zeros((num_nodes, scgpt_dim))
for gene_id, emb in scgpt_dict.items():
    embedding_matrix[gene_id] = torch.tensor(emb, dtype=torch.float32)

### **Dataloaders creation**

In [30]:
perturbations = sorted(pert for pert in adata.obs["target_gene"].unique() if pert != "non-targeting")

# Merge config dictionaries for the model/dataloaders
model_config = {**config['model'], **config['training']}
model_config['dataset_size'] = adata.shape[0]
model_config['pert_to_idx'] = {pert: i for i, pert in enumerate(perturbations)}

In [32]:
from spectra.data import build_model_dataloaders_from_perts_list

train_loader, val_loader, test_loader, _, _, _, train_adata, _, test_adata = build_model_dataloaders_from_perts_list(adata, model_config, config['data']['split_path'])

Total Unique Perturbations: 1251
Train set: 878 perts | 111510 cells
Val set: 121 perts | 14766 cells
Test set: 252 perts | 30809 cells


In [16]:
train_loader, val_loader, _, _, _, _, train_adata, _, _ = build_model_dataloaders_perts_split(adata, model_config)

Total Unique Perturbations: 131
Train set: 92 perts | 121883 cells
Val set: 13 perts | 11746 cells
Test set: 26 perts | 30421 cells


In [36]:
# Load WMSE Weights
weights_path = config['data']['gene_weights_path']
if os.path.exists(weights_path):
    print('found already existing gene weights dictionary! Loading...')
    with open(weights_path, 'rb') as f:
        gene_weights = pickle.load(f)
else:
    print('No gene weights dictionary found. Calculating...')
    gene_weights = compute_weights_enhanced(adata, gene_to_idx, cells_per_pert=128)
    with open(weights_path, 'wb') as f:
        pickle.dump(gene_weights, f)

No gene weights dictionary found. Calculating...


KeyboardInterrupt: 

### **Model initialization**

In [18]:
# Initialize W&B
wandb_support = config['project']['use_wandb']
if wandb_support:
    import wandb
    wandb.login()
    wandb.init(project=config['project']['name'], config=model_config)

In [25]:
model = SPECTRA(
    edge_index=edge_index, 
    num_nodes=num_nodes, 
    device=device, 
    config=model_config,
    gene_embeddings=embedding_matrix,
    gene_weights=gene_weights
).to(device)
print(model)

number of trainable parameters: 76481
SPECTRA(
  (gene_embeddings): Embedding(4922, 512)
  (project_gene): Linear(in_features=512, out_features=64, bias=True)
  (film_layer): GeneExpressionFiLM(
    (scale): Linear(in_features=1, out_features=64, bias=True)
    (shift): Linear(in_features=1, out_features=64, bias=True)
  )
  (add_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (ko_mlp): MLP(
    (network): Sequential(
      (0): Dropout(p=0.0, inplace=False)
      (1): Linear(in_features=512, out_features=64, bias=True)
      (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (encoder): VariationalGraphEncoder(
    (conv1): DirFAGCNConv(64)
    (conv2): DirFAGCNConv(64)
    (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (proj_mu): Linear(in_features=64, out_features=64, bias=True)
    (proj_logstd): Linear(in_features=64, out_features=64, bias=Tru

### **Training**

In [ ]:
idx_to_gene = {v: k for k, v in gene_to_idx.items()}
train(
    model=model, 
    train_loader=train_loader, 
    test_loader=val_loader,
    device=device,
    wandb_support=wandb_support,
    var_names=adata.var_names.tolist(),
    idx_to_gene=idx_to_gene,
    patience=7,
)

In [ ]:
idx_to_gene = {v: k for k, v in gene_to_idx.items()}
train(
    model=model, 
    train_loader=val_loader, 
    test_loader=val_loader,
    lr=model_config['lr'], 
    n_epochs=model_config['n_epochs'],  
    device=device,
    wandb_support=wandb_support,
    var_names=adata.var_names.tolist(),
    idx_to_gene=idx_to_gene,
    alpha_weight=model_config['alpha'],
    beta_weight=model_config['beta'],
    gamma_weight=model_config['gamma'],
    eta_weight=model_config['eta']
)

In [22]:
# exit
if wandb_support:
    wandb.finish()
print('model trained and ready to go! Enjoy!')

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▂▄▅▇█
train/cosine_loss,█▄▂▂▁▁
train/global_loss,█▃▂▁▁▁
train/kl_divergence,▁▅▆▇▇█
train/mmd_ctrl,▁▁▁▁▁▁
train/mmd_pert,█▁▁▁▁▁
train/mse,█▂▂▁▁▁
val/AUPRC,▁
val/test_MMD,█▅▄▂▂▁
val/test_WMSE,▁▂▇█▂▄
epoch,6


model trained and ready to go! Enjoy!
